In [ ]:
# ==============================================================================
# CELL 1 / PHASE 1: ICPSR SUBJECT THESAURUS
# 
# ARCHITECTURE NOTE: ICPSR is a monolingual social science thesaurus. 
# It lacks a bulk Linked Open Data export or public API. This script relies 
# on targeted web scraping (Strategy B) using BeautifulSoup.
#
# SSSOM ALIGNMENT STRATEGY: 
# The script recursively extracts downward from a verified seed. It dynamically
# builds absolute paths by climbing "Broader Terms" links, actively drops 
# "Related Terms" to maintain strict taxonomic boundaries, and leaves 
# Formal_Label and Crosswalks blank based on source architecture.
# ==============================================================================

import os
import sys
import re
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup
from dotenv import load_dotenv

# --- 1. Environment & Central Schema Setup ---
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
config_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "config"))
raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
os.makedirs(raw_data_dir, exist_ok=True)

load_dotenv(os.path.join(config_dir, ".env")) 
sys.path.append(config_dir)
try:
    from ingest_schema_manager import get_registry_info, finalize_row, COLUMN_ORDER
except ImportError:
    raise ImportError("CRITICAL: Could not find ingest_schema_manager.py. Check PYTHONPATH.")

# --- 2. Registry Lookup ---
SOURCE_PREFIX = "ICPSR"
registry_data = get_registry_info(
    prefix=SOURCE_PREFIX, 
    config_dir=config_dir, 
    fallback_name="ICPSR Subject Thesaurus",
    fallback_uri="" # left blank for synthetic CURIE sources
)
SOURCE_NAME = registry_data['Source_Name']
raw_ingest_file = os.path.join(raw_data_dir, f"raw_{SOURCE_PREFIX.lower()}.csv")

DOMAIN = "https://www.icpsr.umich.edu"
BASE_URL = f"{DOMAIN}/web/ICPSR/thesaurus/10001/terms"
HEADERS = {'User-Agent': 'ReligiousMappingProject/1.0', 'Accept': 'text/html'}
SESSION = requests.Session()

TARGET_SEEDS = [
    "26990",  # religion (Original Root)
    
    # --- Lateral Discoveries (Round 1) ---
    "24351",  # Bible
    "24207",  # anti-Semitism
    "24228",  # Arab Israeli relations
    "25693",  # Holocaust
    "25945",  # Jewish peoples
    "26254",  # Middle East
    "27006",  # religious orthodoxy
    "27403",  # synagogues
    "27772",  # Zionism
    "24572",  # church membership
    "26882",  # Protestants
    "26994",  # religious beliefs
    "27000",  # religious fundamentalism
    "27009",  # religious right
    "24378",  # Black Muslims
    "25935",  # Islamic law
    "26011",  # Koran
    "26348",  # Muslims
    "24561",  # Christian Coalition
    "24573",  # church state separation
    "24620",  # clergy
    "24954",  # demographic characteristics
    "26320",  # Moral Majority
    "26759",  # prayer
    "26991",  # religious affiliation
    "26992",  # religious attitudes
    "26993",  # religious behavior
    "26996",  # religious conversion
    "26997",  # religious denominations
    "26998",  # religious doctrines
    "26999",  # religious education
    "27004",  # religious movements
    "27005",  # religious organizations
    "27007",  # religious persecution
    "27008",  # religious reform
    "27078",  # sacred texts
    "27106",  # school prayer
    "27130",  # secularism
    "27764",  # worship
    "24500",  # Catholic Church
    "24503",  # Catholics

    # --- Lateral Discoveries (Round 2) ---
    "24570",  # church buildings
    "26995",  # religious congregations
    "27010",  # religious schools
    "25946",  # Jews
    "27002",  # religious knowledge
    "24569",  # church attendance

    # --- Lateral Discoveries (Round 3) ---
    "26330"   # mosques
]

# --- 3. Persistent Tracking ---
if os.path.exists(raw_ingest_file):
    existing_df = pd.read_csv(raw_ingest_file, dtype={'Concept_ID': str})
    if list(existing_df.columns) != COLUMN_ORDER:
        print(f"[!] Schema mismatch. Deleting {os.path.basename(raw_ingest_file)}...")
        os.remove(raw_ingest_file)
        processed_ids = set()
    else:
        processed_ids = set(existing_df['Concept_ID'].tolist())
        print(f"[*] Resuming: Found {len(processed_ids)} ICPSR concepts already saved.")
else:
    processed_ids = set()

path_cache = {}

# --- 4. Helper Functions ---
def fetch_html_with_retry(url):
    """Executes a 3-try retry loop with exponential backoff."""
    for attempt in range(3):
        try:
            res = SESSION.get(url, headers=HEADERS, timeout=10)
            if res.status_code == 200:
                return res.text
            elif res.status_code in [429, 500, 502, 503, 504]:
                print(f"    [!] Server error {res.status_code}. Retrying in {2 ** attempt}s...")
            else:
                print(f"    [!] HTTP {res.status_code} for {url}. Skipping.")
                return None
        except requests.exceptions.RequestException as e:
            print(f"    [!] Request failed: {e}. Retrying in {2 ** attempt}s...")
        time.sleep(2 ** attempt)
    return None

def extract_table_links(soup, header_text):
    """Locates an h2 header by text and extracts IDs and labels from the adjacent table."""
    header = soup.find('h2', string=re.compile(header_text, re.I))
    if not header: return []
    table = header.find_next_sibling('table')
    if not table: return []
    
    links = table.find_all('a')
    results = []
    for a in links:
        href = a.get('href', '')
        match = re.search(r'terms/(\d+)', href)
        if match:
            results.append({'id': match.group(1), 'label': a.text.strip()})
    return results

def get_icpsr_path(cid, current_label):
    """Recursively fetches broader terms to construct an absolute bottom-up hierarchy path."""
    if cid in path_cache: return path_cache[cid]
    
    html = fetch_html_with_retry(f"{BASE_URL}/{cid}")
    if not html: return current_label

    soup = BeautifulSoup(html, 'html.parser')
    broader_terms = extract_table_links(soup, "Broader Terms")
    
    if not broader_terms:
        path_cache[cid] = current_label
        return current_label
    
    # In case of polyhierarchy, default to the first broader term for visual path rendering
    primary_parent = broader_terms[0]
    parent_path = get_icpsr_path(primary_parent['id'], primary_parent['label'])
    
    full_path = f"{parent_path} > {current_label}"
    path_cache[cid] = full_path
    return full_path

def process_node(cid):
    """Extracts node metadata and recursively crawls narrower terms."""
    if cid in processed_ids: return

    url = f"{BASE_URL}/{cid}"
    html = fetch_html_with_retry(url)
    if not html: return

    soup = BeautifulSoup(html, 'html.parser')
    
    # Primary Label
    h1 = soup.find('h1')
    if not h1: return
    primary_label = h1.text.strip()
    
    print(f"\r[Ingesting] {primary_label[:40]:<40} (ID: {cid})", end="", flush=True)

    # Synonyms (Non-Preferred Terms)
    synonyms = []
    np_header = soup.find('h2', string=re.compile("Non-Preferred Term", re.I))
    if np_header:
        np_table = np_header.find_next_sibling('table')
        if np_table and "No non-preferred terms found" not in np_table.text:
            synonyms = [td.text.strip() for td in np_table.find_all('td') if td.text.strip()]

    # Scope Notes (Description)
    description = ""
    scope_header = soup.find('h2', string=re.compile("Scope Note", re.I))
    if scope_header:
        scope_table = scope_header.find_next_sibling('table')
        if scope_table:
            description = scope_table.text.strip()
            description = re.sub(r'\s+', ' ', description)

    # Parent Resolution
    broader_terms = extract_table_links(soup, "Broader Terms")
    parent_ids = " | ".join([p['id'] for p in broader_terms])

    # Hierarchy Path
    hierarchy_path = get_icpsr_path(cid, primary_label)

    # Write to standard schema
    extracted_data = {
        'Source_System': SOURCE_NAME, 
        'Primary_Label': primary_label,
        'CURIE': f"{SOURCE_PREFIX}:{cid}", 
        'Formal_Label': "", 
        'Concept_Type': 'skos:Concept',
        'Hierarchy_Path': hierarchy_path, 
        'Synonyms': " | ".join(synonyms),
        'Description': description, 
        'Parent_IDs': parent_ids,
        'Concept_ID': str(cid), 
        'URI': url, 
        'Has_Translation': "",
        'Status': 'active',
        'Crosswalks': "" 
    }
    
    clean_row = finalize_row(extracted_data)
    pd.DataFrame([clean_row])[COLUMN_ORDER].to_csv(
        raw_ingest_file, mode='a', index=False, 
        header=not os.path.isfile(raw_ingest_file), encoding='utf-8-sig'
    )
    processed_ids.add(cid)

    # Recurse Downward into Narrower Terms
    narrower_terms = extract_table_links(soup, "Narrower Terms")
    for child in narrower_terms:
        time.sleep(0.5) # API politeness between recursive calls
        process_node(child['id'])

# --- 5. Execution ---
print(f"Starting {SOURCE_PREFIX} Primary Ingestion...\n")
for seed_id in TARGET_SEEDS:
    process_node(seed_id)

print(f"\n\n[COMPLETE] Extracted {len(processed_ids)} ICPSR records.")

In [ ]:
# ==============================================================================
# CELL 2: ICPSR LATERAL DISCOVERY (Hierarchy-Aware)
# Run this cell to harvest "Related Terms" outside the current extracted scope.
# ==============================================================================

import os
import re
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup
from dotenv import load_dotenv

RUN_LATERAL_DISCOVERY = True

if RUN_LATERAL_DISCOVERY:
    notebook_dir = os.path.dirname(os.path.abspath("__file__"))
    raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
    interim_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "interim"))
    
    raw_icpsr_file = os.path.join(raw_data_dir, "raw_icpsr.csv")
    candidates_file = os.path.join(interim_data_dir, "icpsr_lateral_candidates.csv")
    
    DOMAIN = "https://www.icpsr.umich.edu"
    BASE_URL = f"{DOMAIN}/web/ICPSR/thesaurus/10001/terms"
    HEADERS = {'User-Agent': 'ReligiousMappingProject/1.0', 'Accept': 'text/html'}
    SESSION = requests.Session()

    # --- 1. Load Baseline & Inbox ---
    if not os.path.exists(raw_icpsr_file):
        raise FileNotFoundError("Please run Cell 1 to generate raw_icpsr.csv first.")
        
    existing_df = pd.read_csv(raw_icpsr_file, dtype={'Concept_ID': str})
    processed_ids = set(existing_df['Concept_ID'].tolist())
    print(f"[*] Loaded {len(processed_ids)} existing ICPSR concepts for deduplication.")

    known_candidates = set()
    if os.path.exists(candidates_file):
        cand_df = pd.read_csv(candidates_file, dtype={'Candidate_ID': str})
        if 'Candidate_ID' in cand_df.columns:
            known_candidates = set(cand_df['Candidate_ID'].tolist())
            print(f"[*] Loaded {len(known_candidates)} previous candidates from Inbox.")

    # --- 2. Helper Functions ---
    def extract_table_links(soup, header_text):
        header = soup.find('h2', string=re.compile(header_text, re.I))
        if not header: return []
        table = header.find_next_sibling('table')
        if not table: return []
        return [{'id': re.search(r'terms/(\d+)', a['href']).group(1), 'label': a.text.strip()} 
                for a in table.find_all('a') if re.search(r'terms/(\d+)', a.get('href', ''))]

    # --- 3. Harvest Related Concepts ---
    new_candidates = []
    
    print("\nScanning extracted concepts for lateral associations...\n")
    for i, cid in enumerate(list(processed_ids), 1):
        print(f"\r[{i}/{len(processed_ids)}] Checking relations for ID: {cid:<15}", end="", flush=True)
        
        try:
            res = SESSION.get(f"{BASE_URL}/{cid}", headers=HEADERS, timeout=10)
            if res.status_code != 200: continue
                
            soup = BeautifulSoup(res.text, 'html.parser')
            related_terms = extract_table_links(soup, "Related Terms")
            
            for r_node in related_terms:
                r_id = r_node['id']
                
                # Deduplicate against baseline AND existing Inbox
                if r_id not in processed_ids and r_id not in known_candidates:
                    if not any(d['Candidate_ID'] == r_id for d in new_candidates):
                        
                        # We append basic data here. In a deeper workflow, we could call get_icpsr_path 
                        # to provide context, but preserving API quota is prioritized during discovery.
                        new_candidates.append({
                            'Candidate_ID': r_id, 
                            'Candidate_Label': r_node['label'],
                            'Source_Parent_ID': cid
                        })
        except Exception:
            pass
        time.sleep(0.5)

    # --- 4. Export and Append ---
    print(f"\n\nDiscovery Complete! Found {len(new_candidates)} net-new candidates.")
    if new_candidates:
        new_df = pd.DataFrame(new_candidates)
        new_df.to_csv(candidates_file, mode='a', index=False, 
                      header=not os.path.isfile(candidates_file), encoding='utf-8-sig')
        print(f"New candidates appended to: {candidates_file}")
        print("\nNext Step: Review the CSV and select appropriate 'Candidate_ID's to add to TARGET_SEEDS in Cell 1.")
else:
    print("Lateral Discovery is disabled. Set RUN_LATERAL_DISCOVERY = True to execute.")